# Q-Shield: Siamese Network Training for Quishing Detection

**Phase 1:** Contrastive pretraining — learn QR embeddings that separate benign from malicious  
**Phase 2:** Supervised fine-tuning — binary classifier on learned embeddings  

---
**Author:** Nicolas A. Llerena Silva (UTEC)  
**Backbone:** MobileNetV2 (2.9M params)  
**Loss:** Contrastive Loss (Chopra et al., 2005)  
**Datasets:** Trad et al. (9,987 QRs) + CIC Trap4Phish (sampled)

### Google Drive — What to Upload
```
MyDrive/
  QShield/
    QuishingDataset.zip              <- Trad dataset (9 MB) [REQUIRED]
    CIC_QR_sample_1000.zip           <- CIC sample (1 MB)  [OPTIONAL]
```
**Total upload: 9 MB minimum.** Only `QuishingDataset.zip` is mandatory.  
The CIC zip auto-extracts into benign/malicious folders when present.

In [ ]:
# ============================================================
# 0. MOUNT DRIVE & SETUP
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/QShield'
    # Enable GPU: Runtime -> Change runtime type -> T4 GPU
    !nvidia-smi
else:
    BASE = '.'

print(f'Base directory: {BASE}')
print(f'Contents: {os.listdir(BASE)}')

In [ ]:
# ============================================================
# 0.1 INSTALL DEPENDENCIES
# ============================================================
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm

In [ ]:
# ============================================================
# 0.2 IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.manifold import TSNE
from PIL import Image
from pathlib import Path
import pickle, zipfile, random, time, copy
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## 1. MODEL DEFINITION

We define the full Siamese architecture inline so the notebook is self-contained.

In [ ]:
# ============================================================
# 1.1 MobileNetV2 EMBEDDING BACKBONE
# ============================================================

class MobileNetV2Embedding(nn.Module):
    """MobileNetV2 -> 128-d L2-normalized embedding."""

    def __init__(self, embedding_dim=128, pretrained=True):
        super().__init__()
        mobilenet = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        )
        original_conv = mobilenet.features[0][0]
        self.features = mobilenet.features
        # 1-channel input (grayscale QR)
        self.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(
                    original_conv.weight.mean(dim=1, keepdim=True)
                )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.projection(x)
        return F.normalize(x, p=2, dim=1)


class SiameseQRNet(nn.Module):
    """Siamese Network with shared MobileNetV2 backbone."""

    def __init__(self, embedding_dim=128, pretrained=True):
        super().__init__()
        self.backbone = MobileNetV2Embedding(embedding_dim, pretrained)

    def forward_one(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        return self.backbone(x1), self.backbone(x2)


class ContrastiveLoss(nn.Module):
    """Contrastive Loss: pull same-class pairs together, push different-class apart."""

    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        dist = F.pairwise_distance(emb1, emb2)
        loss = (1 - label) * 0.5 * dist.pow(2) + \
               label * 0.5 * F.relu(self.margin - dist).pow(2)
        return loss.mean()


# Quick check
model = SiameseQRNet(embedding_dim=128, pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Siamese model loaded: {total_params:,} parameters')
print(f'Device: {next(model.parameters()).device}')

---
## 2. LOAD DATASETS

In [ ]:
# ============================================================
# 2.1 TRAD ET AL. DATASET (9,987 binary 69x69 matrices)
# ============================================================
trad_zip = os.path.join(BASE, 'QuishingDataset.zip')
trad_dir = os.path.join(BASE, 'trad_data')

if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    with zipfile.ZipFile(trad_zip, 'r') as z:
        z.extractall(trad_dir)
    print('Extracted Trad dataset')

with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

print(f'Trad dataset: {trad_qr.shape}, dtype={trad_qr.dtype}')
print(f'Labels: {np.unique(trad_labels, return_counts=True)}')

In [ ]:
# ============================================================
# 2.2 CIC DATASET (PNG images — sampled)
# ============================================================
cic_benign_dir = os.path.join(BASE, 'CIC_QR_benign_sample')
cic_malicious_dir = os.path.join(BASE, 'CIC_QR_malicious_sample')

# Auto-extract CIC zip if present
cic_zip = os.path.join(BASE, 'CIC_QR_sample_1000.zip')
if os.path.exists(cic_zip) and not os.path.exists(cic_benign_dir):
    print('Extracting CIC sample zip...')
    with zipfile.ZipFile(cic_zip, 'r') as z:
        z.extractall(BASE)
    print('Done.')

HAS_CIC = os.path.exists(cic_benign_dir) and os.path.exists(cic_malicious_dir)

if HAS_CIC:
    cic_benign_files = sorted(Path(cic_benign_dir).glob('*.png'))
    cic_malicious_files = sorted(Path(cic_malicious_dir).glob('*.png'))
    print(f'CIC benign: {len(cic_benign_files)} images')
    print(f'CIC malicious: {len(cic_malicious_files)} images')
else:
    print('CIC sample not found — training on Trad dataset only (this is fine)')
    print('To add CIC later: upload CIC_QR_sample_1000.zip to Drive/QShield/')

---
## 3. PAIR DATASETS FOR CONTRASTIVE LEARNING

In [ ]:
# ============================================================
# 3.1 TRAD PAIR DATASET
# ============================================================

class TradPairDataset(Dataset):
    """Generates pairs from Trad 69x69 binary arrays."""

    def __init__(self, qr_arrays, labels, pairs_per_epoch=20000):
        self.qr = qr_arrays.astype(np.float32)
        self.labels = np.array(labels)
        self.pairs_per_epoch = pairs_per_epoch
        self.idx_0 = np.where(self.labels == 0)[0]
        self.idx_1 = np.where(self.labels == 1)[0]

    def __len__(self):
        return self.pairs_per_epoch

    def _to_tensor(self, idx):
        arr = self.qr[idx]  # (69, 69), values 0 or 1
        t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # (1, 1, 69, 69)
        t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
        return t.squeeze(0)  # (1, 224, 224)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c1 = random.choice([0, 1])
        i1 = random.choice(self.idx_0 if c1 == 0 else self.idx_1)

        if same:
            i2 = random.choice(self.idx_0 if c1 == 0 else self.idx_1)
            pair_label = 0.0  # same class
        else:
            i2 = random.choice(self.idx_1 if c1 == 0 else self.idx_0)
            pair_label = 1.0  # different class

        return self._to_tensor(i1), self._to_tensor(i2), torch.tensor(pair_label)


class CICPairDataset(Dataset):
    """Generates pairs from CIC PNG images."""

    def __init__(self, benign_files, malicious_files, pairs_per_epoch=20000):
        self.files = {0: benign_files, 1: malicious_files}
        self.pairs_per_epoch = pairs_per_epoch

    def __len__(self):
        return self.pairs_per_epoch

    def _load(self, cls, idx):
        img = Image.open(self.files[cls][idx]).convert('L').resize((224, 224))
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).unsqueeze(0)  # (1, 224, 224)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c1 = random.choice([0, 1])
        i1 = random.randint(0, len(self.files[c1]) - 1)

        if same:
            c2 = c1
            pair_label = 0.0
        else:
            c2 = 1 - c1
            pair_label = 1.0

        i2 = random.randint(0, len(self.files[c2]) - 1)
        return self._load(c1, i1), self._load(c2, i2), torch.tensor(pair_label)


# ============================================================
# 3.2 CLASSIFICATION DATASET (for Phase 2)
# ============================================================

class TradClassificationDataset(Dataset):
    """Single-image dataset for supervised fine-tuning."""

    def __init__(self, qr_arrays, labels):
        self.qr = qr_arrays.astype(np.float32)
        self.labels = np.array(labels, dtype=np.float32)

    def __len__(self):
        return len(self.qr)

    def __getitem__(self, idx):
        arr = self.qr[idx]
        t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False)
        return t.squeeze(0), torch.tensor(self.labels[idx])


print('Dataset classes defined.')

In [ ]:
# ============================================================
# 3.3 CREATE TRAIN/VAL SPLITS
# ============================================================
from sklearn.model_selection import train_test_split

# Trad: 80/20 stratified split
idx_train, idx_val = train_test_split(
    np.arange(len(trad_labels)), test_size=0.2,
    stratify=trad_labels, random_state=SEED
)

trad_qr_train = trad_qr[idx_train]
trad_labels_train = trad_labels[idx_train]
trad_qr_val = trad_qr[idx_val]
trad_labels_val = trad_labels[idx_val]

print(f'Train: {len(idx_train)} (benign={sum(trad_labels_train==0)}, phishing={sum(trad_labels_train==1)})')
print(f'Val:   {len(idx_val)} (benign={sum(trad_labels_val==0)}, phishing={sum(trad_labels_val==1)})')

# Create pair datasets
PAIRS_TRAIN = 20000  # pairs per epoch
PAIRS_VAL = 4000
BATCH_SIZE = 32

train_pair_ds = TradPairDataset(trad_qr_train, trad_labels_train, PAIRS_TRAIN)
val_pair_ds = TradPairDataset(trad_qr_val, trad_labels_val, PAIRS_VAL)

train_pair_loader = DataLoader(train_pair_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=True)
val_pair_loader = DataLoader(val_pair_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f'\nTrain loader: {len(train_pair_loader)} batches x {BATCH_SIZE}')
print(f'Val loader:   {len(val_pair_loader)} batches x {BATCH_SIZE}')

---
## 4. PHASE 1: SIAMESE CONTRASTIVE PRETRAINING

In [ ]:
# ============================================================
# 4.1 TRAINING CONFIGURATION
# ============================================================
EMBEDDING_DIM = 128
MARGIN = 2.0
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS_PHASE1 = 25

model = SiameseQRNet(embedding_dim=EMBEDDING_DIM, pretrained=True).to(device)
criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1)

print(f'Phase 1 config:')
print(f'  Embedding dim: {EMBEDDING_DIM}')
print(f'  Margin: {MARGIN}')
print(f'  LR: {LR}')
print(f'  Epochs: {EPOCHS_PHASE1}')
print(f'  Pairs/epoch: {PAIRS_TRAIN} train, {PAIRS_VAL} val')

In [ ]:
# ============================================================
# 4.2 TRAINING LOOP — PHASE 1
# ============================================================

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
best_model_state = None

print(f'\n{"Epoch":>5} | {"Train Loss":>10} | {"Val Loss":>10} | {"Train Acc":>10} | {"Val Acc":>10} | {"LR":>10}')
print('-' * 70)

for epoch in range(1, EPOCHS_PHASE1 + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for img1, img2, labels in train_pair_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

        optimizer.zero_grad()
        emb1, emb2 = model(img1, img2)
        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * img1.size(0)
        with torch.no_grad():
            dist = F.pairwise_distance(emb1, emb2)
            pred = (dist > MARGIN / 2).float()
            train_correct += (pred == labels).sum().item()
            train_total += labels.size(0)

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for img1, img2, labels in val_pair_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            emb1, emb2 = model(img1, img2)
            loss = criterion(emb1, emb2, labels)
            val_loss += loss.item() * img1.size(0)
            dist = F.pairwise_distance(emb1, emb2)
            pred = (dist > MARGIN / 2).float()
            val_correct += (pred == labels).sum().item()
            val_total += labels.size(0)

    scheduler.step()

    # Metrics
    tl = train_loss / train_total
    vl = val_loss / val_total
    ta = train_correct / train_total
    va = val_correct / val_total

    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)

    # Save best
    if vl < best_val_loss:
        best_val_loss = vl
        best_model_state = copy.deepcopy(model.state_dict())
        marker = ' *'
    else:
        marker = ''

    lr_now = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    print(f'{epoch:>5} | {tl:>10.4f} | {vl:>10.4f} | {ta:>9.1%} | {va:>9.1%} | {lr_now:>10.6f}{marker}')

# Restore best model
model.load_state_dict(best_model_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

# Save Phase 1 checkpoint
ckpt_path = os.path.join(BASE, 'siamese_phase1_best.pth')
torch.save(best_model_state, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')

In [ ]:
# ============================================================
# 4.3 TRAINING CURVES
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Contrastive Loss')
axes[0].set_title('Phase 1: Contrastive Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pair Accuracy')
axes[1].set_title('Phase 1: Pair Classification Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Siamese Contrastive Pretraining — Trad Dataset', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'phase1_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. EMBEDDING VISUALIZATION (t-SNE)

In [ ]:
# ============================================================
# 5.1 EXTRACT EMBEDDINGS FOR ALL VALIDATION SAMPLES
# ============================================================
val_cls_ds = TradClassificationDataset(trad_qr_val, trad_labels_val)
val_cls_loader = DataLoader(val_cls_ds, batch_size=64, shuffle=False, num_workers=2)

model.eval()
all_embeddings = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_cls_loader, desc='Extracting embeddings'):
        images = images.to(device)
        emb = model.forward_one(images)
        all_embeddings.append(emb.cpu().numpy())
        all_labels.append(labels.numpy())

embeddings = np.concatenate(all_embeddings)
labels_arr = np.concatenate(all_labels)
print(f'Embeddings shape: {embeddings.shape}')

# t-SNE
print('Running t-SNE...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
tsne_result = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before training (random) vs after training
for label, color, name in [(0, '#2ecc71', 'Benign'), (1, '#e74c3c', 'Phishing')]:
    mask = labels_arr == label
    axes[1].scatter(tsne_result[mask, 0], tsne_result[mask, 1],
                    c=color, label=name, alpha=0.5, s=15, edgecolors='none')

axes[1].set_title('After Siamese Pretraining', fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(markerscale=3)

# Random embeddings for comparison
random_emb = np.random.randn(*embeddings.shape)
tsne_random = TSNE(n_components=2, random_state=SEED).fit_transform(random_emb)
for label, color, name in [(0, '#2ecc71', 'Benign'), (1, '#e74c3c', 'Phishing')]:
    mask = labels_arr == label
    axes[0].scatter(tsne_random[mask, 0], tsne_random[mask, 1],
                    c=color, label=name, alpha=0.5, s=15, edgecolors='none')
axes[0].set_title('Random Embeddings (Baseline)', fontweight='bold')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].legend(markerscale=3)

fig.suptitle('Siamese Embedding Space — Trad Dataset (Validation)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'phase1_tsne_embeddings.png'), dpi=300, bbox_inches='tight')
plt.show()
print('If clusters are separable, contrastive pretraining succeeded.')

---
## 6. PHASE 2: SUPERVISED CLASSIFICATION

In [ ]:
# ============================================================
# 6.1 CLASSIFICATION HEAD
# ============================================================

class QRClassifier(nn.Module):
    """Binary classifier on top of pretrained Siamese backbone."""

    def __init__(self, backbone, embedding_dim=128, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        emb = self.backbone(x)
        return self.head(emb)


# Build classifier with pretrained backbone
classifier = QRClassifier(
    backbone=model.backbone,
    embedding_dim=EMBEDDING_DIM,
    freeze_backbone=False  # fine-tune everything
).to(device)

cls_params = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f'Classifier parameters (trainable): {cls_params:,}')

In [ ]:
# ============================================================
# 6.2 PHASE 2 TRAINING
# ============================================================
EPOCHS_PHASE2 = 15
LR_PHASE2 = 5e-5

train_cls_ds = TradClassificationDataset(trad_qr_train, trad_labels_train)
val_cls_ds = TradClassificationDataset(trad_qr_val, trad_labels_val)

train_cls_loader = DataLoader(train_cls_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_cls_loader = DataLoader(val_cls_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

cls_criterion = nn.BCEWithLogitsLoss()
cls_optimizer = optim.Adam(classifier.parameters(), lr=LR_PHASE2, weight_decay=1e-4)
cls_scheduler = optim.lr_scheduler.CosineAnnealingLR(cls_optimizer, T_max=EPOCHS_PHASE2)

cls_history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
best_val_auc = 0
best_cls_state = None

print(f'\n{"Ep":>3} | {"Train Loss":>10} | {"Val Loss":>10} | {"Val AUC":>8} | {"Val F1":>8} | {"Val Prec":>8} | {"Val Rec":>8}')
print('-' * 80)

for epoch in range(1, EPOCHS_PHASE2 + 1):
    # --- Train ---
    classifier.train()
    train_loss = 0
    for images, labels in train_cls_loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        cls_optimizer.zero_grad()
        logits = classifier(images)
        loss = cls_criterion(logits, labels)
        loss.backward()
        cls_optimizer.step()
        train_loss += loss.item() * images.size(0)

    # --- Validate ---
    classifier.eval()
    val_loss = 0
    all_probs, all_true = [], []

    with torch.no_grad():
        for images, labels in val_cls_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            logits = classifier(images)
            loss = cls_criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_true.extend(labels.cpu().numpy().flatten())

    cls_scheduler.step()

    all_probs = np.array(all_probs)
    all_true = np.array(all_true)
    preds = (all_probs >= 0.5).astype(int)

    tl = train_loss / len(train_cls_ds)
    vl = val_loss / len(val_cls_ds)
    auc = roc_auc_score(all_true, all_probs)
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(all_true, preds)
    prec = precision_score(all_true, preds)
    rec = recall_score(all_true, preds)

    cls_history['train_loss'].append(tl)
    cls_history['val_loss'].append(vl)
    cls_history['val_auc'].append(auc)
    cls_history['val_f1'].append(f1)

    marker = ''
    if auc > best_val_auc:
        best_val_auc = auc
        best_cls_state = copy.deepcopy(classifier.state_dict())
        marker = ' *'

    print(f'{epoch:>3} | {tl:>10.4f} | {vl:>10.4f} | {auc:>7.4f} | {f1:>7.4f} | {prec:>7.4f} | {rec:>7.4f}{marker}')

# Restore best
classifier.load_state_dict(best_cls_state)
print(f'\nBest val AUC: {best_val_auc:.4f}')

# Save Phase 2 checkpoint
ckpt2_path = os.path.join(BASE, 'classifier_phase2_best.pth')
torch.save(best_cls_state, ckpt2_path)
print(f'Checkpoint saved: {ckpt2_path}')

In [ ]:
# ============================================================
# 6.3 FINAL EVALUATION
# ============================================================
classifier.eval()
all_probs, all_true = [], []

with torch.no_grad():
    for images, labels in val_cls_loader:
        images = images.to(device)
        logits = classifier(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs.flatten())
        all_true.extend(labels.numpy().flatten())

all_probs = np.array(all_probs)
all_true = np.array(all_true)
preds = (all_probs >= 0.5).astype(int)

print('='*60)
print(' FINAL RESULTS — Siamese + Classifier (Trad Dataset)')
print('='*60)
print(f'\nAUC-ROC: {roc_auc_score(all_true, all_probs):.4f}')
print(f'\n{classification_report(all_true, preds, target_names=["Benign", "Phishing"])}')

# Confusion matrix
cm = confusion_matrix(all_true, preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign', 'Phishing'], yticklabels=['Benign', 'Phishing'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (AUC={roc_auc_score(all_true, all_probs):.4f})', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(all_true, all_probs)
axes[1].plot(fpr, tpr, linewidth=2, color='#e74c3c')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve (AUC={roc_auc_score(all_true, all_probs):.4f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Q-Shield Siamese Classifier — Final Evaluation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'phase2_final_results.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## 7. COMPARISON TABLE: Our Approach vs Baselines

In [ ]:
# ============================================================
# 7.1 COMPARISON WITH BASELINES
# ============================================================
final_auc = roc_auc_score(all_true, all_probs)
final_f1 = f1_score(all_true, preds)

comparison = pd.DataFrame([
    {'Method': 'Trad et al. (raw pixels + XGBoost)', 'Features': 4761, 'AUC': 0.9133,
     'Explainable': 'Low', 'Source': 'Reported'},
    {'Method': 'Our 25 handcrafted + RF', 'Features': 25, 'AUC': 0.8132,
     'Explainable': 'High (SHAP)', 'Source': 'Our experiment'},
    {'Method': 'Siamese MobileNetV2 (ours)', 'Features': 128, 'AUC': round(final_auc, 4),
     'Explainable': 'Medium (Grad-CAM)', 'Source': 'This notebook'},
])

print('\nMETHOD COMPARISON — Trad Dataset')
print('=' * 80)
print(comparison.to_string(index=False))
print(f'\nSiamese improvement over handcrafted: {(final_auc - 0.8132)/0.8132*100:+.1f}%')
print(f'Gap vs Trad raw-pixel baseline: {(final_auc - 0.9133)/0.9133*100:+.1f}%')

In [ ]:
# ============================================================
# 7.2 SAVE ALL RESULTS
# ============================================================
results = {
    'phase1': history,
    'phase2': cls_history,
    'final_auc': final_auc,
    'final_f1': final_f1,
    'confusion_matrix': cm.tolist(),
    'config': {
        'embedding_dim': EMBEDDING_DIM,
        'margin': MARGIN,
        'lr_phase1': LR,
        'lr_phase2': LR_PHASE2,
        'epochs_phase1': EPOCHS_PHASE1,
        'epochs_phase2': EPOCHS_PHASE2,
        'batch_size': BATCH_SIZE,
        'pairs_per_epoch': PAIRS_TRAIN,
    }
}

import json
with open(os.path.join(BASE, 'experiment_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('All results saved.')
print(f'\nFiles in {BASE}:')
for f in sorted(os.listdir(BASE)):
    if not f.startswith('.'):
        size = os.path.getsize(os.path.join(BASE, f))
        print(f'  {f:<40} {size/1024:.0f} KB')